In [1]:
import pandas as pd
import numpy as np
import joblib
import os

# Load raw data
df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())


Shape: (7043, 21)
Columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [2]:
# Fix TotalCharges — convert to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# For rows where tenure=0, TotalCharges should be 0
df.loc[df['tenure'] == 0, 'TotalCharges'] = 0

# Drop remaining nulls (if any)
df = df.dropna(subset=['TotalCharges'])

# Drop customerID — not useful for prediction
df = df.drop(columns=['customerID'])

# Encode target
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print("Shape after cleaning:", df.shape)
print("Nulls remaining:", df.isnull().sum().sum())
print("Churn distribution:\n", df['Churn'].value_counts())

Shape after cleaning: (7043, 20)
Nulls remaining: 0
Churn distribution:
 Churn
0    5174
1    1869
Name: count, dtype: int64


In [3]:
import matplotlib.pyplot as plt
import seaborn as sns

# Pearson correlation
corr = df['TotalCharges'].corr(df['tenure'])
print(f"Pearson correlation between TotalCharges and tenure: {corr:.4f}")

if corr > 0.8:
    print("Correlation > 0.8 — TotalCharges is redundant, dropping it")
    df = df.drop(columns=['TotalCharges'])
else:
    print("Correlation acceptable — keeping TotalCharges")

print("Shape now:", df.shape)

Pearson correlation between TotalCharges and tenure: 0.8262
Correlation > 0.8 — TotalCharges is redundant, dropping it
Shape now: (7043, 19)


In [4]:
contract_map = {'Month-to-month': 0, 'One year': 1, 'Two year': 2}
df['Contract'] = df['Contract'].map(contract_map)

yes_no_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling',
               'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
               'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

for col in yes_no_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0, 'No phone service': 0, 'No internet service': 0})

df = pd.get_dummies(df, columns=['InternetService', 'PaymentMethod'], drop_first=False)
df['gender'] = df['gender'].map({'Male': 1, 'Female': 0})
print("Shape:", df.shape)
print(df.columns.tolist())

Shape: (7043, 24)
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'MonthlyCharges', 'Churn', 'InternetService_DSL', 'InternetService_Fiber optic', 'InternetService_No', 'PaymentMethod_Bank transfer (automatic)', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']


In [5]:
# Cell 5: Stratified Train/Test Split + scale_pos_weight

from sklearn.model_selection import train_test_split
import numpy as np

# ── 1. Separate features and target ──────────────────────────────────────────
X = df.drop(columns=['Churn'])
y = df['Churn']

print(f"Feature matrix : {X.shape}")
print(f"Target vector  : {y.shape}")
print(f"\nClass distribution:\n{y.value_counts()}")
print(f"Churn rate     : {y.mean():.2%}")

# ── 2. Stratified split (80/20) ───────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # preserves churn ratio in both splits
)

print(f"\nTrain size : {X_train.shape[0]}  |  Churn rate: {y_train.mean():.2%}")
print(f"Test  size : {X_test.shape[0]}   |  Churn rate: {y_test.mean():.2%}")

# ── 3. scale_pos_weight for XGBoost ──────────────────────────────────────────
# Formula: count(negative class) / count(positive class)
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print(f"\nTrain negatives (no churn) : {neg}")
print(f"Train positives (churn)    : {pos}")
print(f"scale_pos_weight           : {scale_pos_weight:.4f}")
# You'll pass this directly to XGBClassifier(scale_pos_weight=...)

Feature matrix : (7043, 23)
Target vector  : (7043,)

Class distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64
Churn rate     : 26.54%

Train size : 5634  |  Churn rate: 26.54%
Test  size : 1409   |  Churn rate: 26.54%

Train negatives (no churn) : 4139
Train positives (churn)    : 1495
scale_pos_weight           : 2.7686


In [8]:
# Cell 6: ColumnTransformer + RobustScaler Pipeline

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer

# ── 1. Identify column types ──────────────────────────────────────────────────
binary_cols = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents',
    'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'PaperlessBilling', 'Contract',
    'InternetService_DSL', 'InternetService_Fiber optic', 'InternetService_No',
    'PaymentMethod_Bank transfer (automatic)',
    'PaymentMethod_Credit card (automatic)',
    'PaymentMethod_Electronic check',
    'PaymentMethod_Mailed check'
]

continuous_cols = ['tenure', 'MonthlyCharges']

print(f"Binary cols     : {len(binary_cols)}")
print(f"Continuous cols : {continuous_cols}")
print(f"Total           : {len(binary_cols) + len(continuous_cols)} (should be 23)")

# ── 2. ColumnTransformer ──────────────────────────────────────────────────────
preprocessor = ColumnTransformer(
    transformers=[
        ('scale', RobustScaler(), continuous_cols),
        ('pass',  'passthrough',  binary_cols)
    ],
    remainder='drop'
)

# ── 3. Fit on train, transform both splits ────────────────────────────────────
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed  = preprocessor.transform(X_test)

print(f"\nX_train_processed shape : {X_train_processed.shape}")
print(f"X_test_processed shape  : {X_test_processed.shape}")

# ── 4. Verify scaling on continuous cols ─────────────────────────────────────
col_order = continuous_cols + binary_cols
processed_df = pd.DataFrame(X_train_processed, columns=col_order)

print("\nContinuous cols after RobustScaler (train):")
print(processed_df[continuous_cols].describe().round(3))

Binary cols     : 21
Continuous cols : ['tenure', 'MonthlyCharges']
Total           : 23 (should be 23)

X_train_processed shape : (5634, 23)
X_test_processed shape  : (1409, 23)

Continuous cols after RobustScaler (train):
          tenure  MonthlyCharges
count   5634.000        5634.000
unique    73.000        1489.000
top       -0.609          -0.928
freq     487.000          50.000


In [9]:
# Cell 7: Verify scaling + Save all artifacts

import numpy as np
import pickle
import os

# ── 1. Fix dtype and verify scaling ──────────────────────────────────────────
processed_df = processed_df.astype(float)

print("Continuous cols after RobustScaler (train):")
print(processed_df[continuous_cols].describe().round(3))

# Median should be ~0, IQR should be ~1 after RobustScaler
print(f"\ntenure         median: {processed_df['tenure'].median():.3f}  (expect ~0)")
print(f"MonthlyCharges median: {processed_df['MonthlyCharges'].median():.3f}  (expect ~0)")

# ── 2. Save artifacts ─────────────────────────────────────────────────────────
os.makedirs('../model/artifacts', exist_ok=True)

# Save preprocessor (replaces old scaler.pkl)
with open('../model/artifacts/preprocessor.pkl', 'wb') as f:
    pickle.dump(preprocessor, f)

# Save processed splits
np.save('../model/artifacts/X_train.npy', X_train_processed)
np.save('../model/artifacts/X_test.npy',  X_test_processed)
np.save('../model/artifacts/y_train.npy', y_train.values)
np.save('../model/artifacts/y_test.npy',  y_test.values)

# Save scale_pos_weight for XGBoost training later
with open('../model/artifacts/scale_pos_weight.pkl', 'wb') as f:
    pickle.dump(scale_pos_weight, f)

# Save column order (critical for API later)
with open('../model/artifacts/feature_columns.pkl', 'wb') as f:
    pickle.dump(col_order, f)

print("\n✅ Artifacts saved:")
for f in os.listdir('../model/artifacts'):
    size = os.path.getsize(f'../model/artifacts/{f}')
    print(f"   {f:<35} {size:>8} bytes")

Continuous cols after RobustScaler (train):
         tenure  MonthlyCharges
count  5634.000        5634.000
mean      0.076          -0.103
std       0.534           0.555
min      -0.630          -0.959
25%      -0.435          -0.641
50%       0.000           0.000
75%       0.565           0.359
max       0.935           0.888

tenure         median: 0.000  (expect ~0)
MonthlyCharges median: 0.000  (expect ~0)

✅ Artifacts saved:
   churn_model.pkl                       303910 bytes
   feature_columns.pkl                      465 bytes
   preprocessor.pkl                        2428 bytes
   scaler.pkl                              2279 bytes
   scale_pos_weight.pkl                     117 bytes
   X_test.npy                             75030 bytes
   X_train.npy                           299176 bytes
   y_test.npy                             11400 bytes
   y_train.npy                            45200 bytes
